<a href="https://colab.research.google.com/github/talkeasy-ai/openWakeWord/blob/main/notebooks/automatic_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup

# install piper-sample-generator (currently only supports linux systems)
# pinned to v2.0.0: newer releases moved generate_samples.py into a package and require an explicit
# `model` argument, which openwakeword/train.py doesn't pass
!rm -rf piper-sample-generator
!git clone --depth 1 --branch v2.0.0 https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
# the generator checkpoint is a fully pickled model; PyTorch 2.6+ defaults torch.load to weights_only=True, which rejects it
!sed -i 's/torch.load(model_path)/torch.load(model_path, weights_only=False)/' piper-sample-generator/generate_samples.py
# piper-phonemize has no wheels for Python 3.13+; this fork provides the same `piper_phonemize` module
!pip install piper-phonemize-fix==1.2.2
# maintained fork of webrtcvad (same import name); the original imports pkg_resources
!pip install webrtcvad-wheels

# install openwakeword from this fork (it carries the fixes for current Colab torch/torchaudio)
!rm -rf openwakeword
!git clone --depth 1 https://github.com/talkeasy-ai/openWakeWord openwakeword
# a PyPI openwakeword in dist-packages takes precedence over the editable install and lacks the training code
!pip uninstall -y -q openwakeword
# --no-deps: speexdsp-ns (optional noise suppression) has no wheels for Python 3.13+ and would abort the install
!pip install --no-deps -e ./openwakeword
!pip install 'onnxruntime>=1.10.0,<2' 'ai-edge-litert>=2.0.2,<3' 'tqdm>=4.0,<5.0' 'scipy>=1.3,<2' 'scikit-learn>=1,<2' 'requests>=2.0,<3'

# install other dependencies (Colab's preinstalled torch/torchaudio are kept)
!pip install mutagen
!pip install torchinfo
!pip install torchmetrics
!pip install speechbrain
!pip install audiomentations
# 0.12.0+: 0.11.0 calls torchaudio.set_audio_backend, which newer torchaudio releases removed
!pip install 'torch-audiomentations>=0.12.0'
!pip install acoustics
!pip install onnx
!pip install pronouncing
!pip install -U datasets pyarrow
!pip install deep-phonemizer
!pip install onnxruntime

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm

# the editable install isn't visible to this already-running kernel; import openwakeword from the clone
sys.path.insert(0, os.path.abspath("openwakeword"))
import openwakeword
assert openwakeword.__file__.startswith(os.path.abspath("openwakeword")), f"stale openwakeword at {openwakeword.__file__}"


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html
import os
import numpy as np
import scipy.io.wavfile
import datasets
import io
import soundfile as sf
from tqdm import tqdm
from huggingface_hub import HfFileSystem

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Load the dataset with streaming (leaving audio casting off to keep URLs intact)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
rir_dataset = rir_dataset.cast_column("audio", datasets.Audio(decode=False))

# Initialize Hugging Face File System
fs = HfFileSystem()

# Save clips to 16-bit PCM wav files by reading directly via the hf:// filesystem
for row in tqdm(rir_dataset):
    url = row['audio']['path']
    name = url.split('/')[-1]

    try:
        # Open the virtual hf:// path directly using HfFileSystem
        with fs.open(url, "rb") as f:
            audio_bytes = f.read()

        # Standardize and save as 16kHz WAV
        data, samplerate = sf.read(io.BytesIO(audio_bytes))
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (data * 32767).astype(np.int16))
    except Exception as e:
        print(f"Skipping {name} due to processing error: {e}")

In [ ]:
import os
import numpy as np
import scipy.io.wavfile
import datasets
from tqdm import tqdm
import soundfile as sf

# AudioSet noise data (parquet format)
output_dir_audioset = "./audioset_16k"
if not os.path.exists(output_dir_audioset):
    os.mkdir(output_dir_audioset)

# Only run if not already populated to save execution time
if len(os.listdir(output_dir_audioset)) < 100:
    print("Loading AudioSet from Parquet files...")
    try:
        ds = datasets.load_dataset("agkphysics/AudioSet", "balanced", split="train", streaming=True)

        print("Extracting and resampling AudioSet files to 16kHz WAV...")
        count = len(os.listdir(output_dir_audioset))
        for row in tqdm(ds, total=100):
            if count >= 100:
                break
            try:
                audio = row["audio"]
                data = audio["array"]
                samplerate = audio["sampling_rate"]

                if len(data.shape) > 1:
                    data = data.mean(axis=1)

                if samplerate != 16000:
                    import scipy.signal
                    num_samples = int(len(data) * 16000 / samplerate)
                    data = scipy.signal.resample(data, num_samples)

                name = f"audioset_{count}.wav"
                scipy.io.wavfile.write(os.path.join(output_dir_audioset, name), 16000, (data * 32767).astype(np.int16))
                count += 1
            except Exception as e:
                continue
        print(f"Successfully extracted {count} AudioSet files.")
    except Exception as e:
        print(f"AudioSet download/extraction failed: {e}")
else:
    print("AudioSet files already prepared.")


In [ ]:
# Download a sample of the Free Music Archive (FMA) as background music
# https://github.com/mdeff/fma
# fma_small.zip is 7.6 GB, so instead of downloading it, read only the needed MP3s out of it with HTTP range requests
import io
import os
import zipfile
import numpy as np
import requests
import scipy.io.wavfile
import scipy.signal
import soundfile as sf
from tqdm import tqdm


class HTTPRangeFile(io.RawIOBase):
    """A read-only, seekable file over HTTP range requests (enough for zipfile)."""
    def __init__(self, url):
        self.url, self.pos, self.session = url, 0, requests.Session()
        self.size = int(self.session.head(url, allow_redirects=True).headers["Content-Length"])

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        self.pos = {io.SEEK_SET: 0, io.SEEK_CUR: self.pos, io.SEEK_END: self.size}[whence] + offset
        return self.pos

    def readinto(self, b):
        if self.pos >= self.size or len(b) == 0:
            return 0
        end = min(self.pos + len(b), self.size) - 1
        r = self.session.get(self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        r.raise_for_status()
        b[:len(r.content)] = r.content
        self.pos += len(r.content)
        return len(r.content)


n_fma_clips = 50
output_dir_fma = "./fma"
os.makedirs(output_dir_fma, exist_ok=True)

if len(os.listdir(output_dir_fma)) < n_fma_clips:
    remote_zip = io.BufferedReader(HTTPRangeFile("https://os.unil.cloud.switch.ch/fma/fma_small.zip"), buffer_size=1 << 20)
    with zipfile.ZipFile(remote_zip) as zf:
        mp3_files = [f for f in zf.namelist() if f.lower().endswith(".mp3")][:n_fma_clips]
        for count, member in enumerate(tqdm(mp3_files, desc="Downloading FMA clips")):
            try:
                data, samplerate = sf.read(io.BytesIO(zf.read(member)))
                if data.ndim > 1:
                    data = data.mean(axis=1)
                if samplerate != 16000:
                    data = scipy.signal.resample_poly(data, 16000, samplerate)
                scipy.io.wavfile.write(os.path.join(output_dir_fma, f"music_{count}.wav"), 16000,
                                       (np.clip(data, -1, 1) * 32767).astype(np.int16))
            except Exception as e:
                print(f"Skipping {member}: {e}")
print("FMA (Music) WAVs:", len(os.listdir(output_dir_fma)))

In [ ]:
# Verify if files exist in the target paths and check for any errors
import glob
print("Audioset WAVs:", len(glob.glob('./audioset_16k/*.wav')))
print("FMA (Music) WAVs:", len(glob.glob('./fma/*.wav')))


In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

In [ ]:
# Modify values in the config and save a new version

# spelled as dictionary words: "talkeasy" isn't in CMUdict, which makes train.py download the DeepPhonemizer
# model, and its download URL is dead. Piper pronounces "talk easy" the same way.
config["target_phrase"] = ["hey talk easy"]
config["model_name"] = "hey_talkeasy"
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file
# Runs train.py in a fresh process, so patched/reinstalled packages are always picked up
import sys

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# Step 2: Augment the generated clips
# Runs train.py in a fresh process, so patched/reinstalled packages are always picked up
import sys

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: Train model
# Runs train.py in a fresh process, so patched/reinstalled packages are always picked up
import sys

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

In [ ]:
# Step 4 (Optional): convert the ONNX model to tflite
# The onnx_tf converter this used is unmaintained and can't import current onnx releases, so it isn't installed and
# this step is skipped. The .onnx model from Step 3 works with openwakeword (inference_framework="onnx").
import os

onnx_path = f"my_custom_model/{config['model_name']}.onnx"
print(f"ONNX model: {onnx_path} (exists: {os.path.exists(onnx_path)})")
try:
    import onnx
    from onnx_tf.backend import prepare
except ImportError as e:
    print(f"Skipping tflite conversion: onnx_tf is not available ({e})")
else:
    import tempfile
    import tensorflow as tf
    tf_rep = prepare(onnx.load(onnx_path), device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        tflite_model = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model")).convert()
    with open(onnx_path[:-len(".onnx")] + ".tflite", "wb") as f:
        f.write(tflite_model)


# Voice match data for TalkEasy

TalkEasy's voice match only accepts the enrolled user if it was also trained on the wake phrase said by *other* voices. Step 5 turns the Piper clips from Step 1 into that set (`<model_name>_other_speakers.f32`), featurized exactly as the app does; Step 6 downloads it with the model. Run them after Step 3.

In [ ]:
# Step 5: Build the voice match "other speakers" set for TalkEasy
#
# Voice match (TalkEasy's per-user speaker filter, lib/services/wake_word/speaker_verifier.dart) trains against
# the wake phrase said by many other voices so it rejects them. The Piper clips generated in Step 1 are ideal:
# ~1,000 clips across the LibriTTS-R speakers. This featurizes them exactly as the app featurizes the user's own
# recordings (lib/services/wake_word/voice_enrollment.dart, scripts/build_verifier_other_speakers.py):
#   - each clip is fed after 25 steps (2 s) of silence, one 1280-sample step at a time
#   - at five step alignments (offsets 0, 256, 512, 768, 1024 samples)
#   - every step the trained model scores >= THRESHOLD keeps its classifier input (16 x 96 embeddings)
# and writes the windows as one little-endian float32 file for the app's wake_word.json "otherSpeakers".
#
# Licensing: the clips come from Piper's en_US-libritts_r-medium voice (CC BY 4.0) -- keep the attribution.
import glob
import os

import random
import numpy as np
import scipy.io.wavfile
import scipy.signal
from openwakeword.model import Model

THRESHOLD = 0.5      # the app manifest's "threshold"
MAX_WINDOWS = 300    # app-side training time grows with the square of the example count
OFFSETS = [0, 256, 512, 768, 1024]
STEP = 1280
WARMUP_STEPS = 25
SR = 16000

model_name = config["model_name"]
output_dir = config.get("output_dir", "./my_custom_model")
classifier = os.path.join(output_dir, f"{model_name}.onnx")
resources = "./openwakeword/openwakeword/resources/models"
oww = Model(
    wakeword_models=[classifier],
    inference_framework="onnx",
    melspec_model_path=os.path.join(resources, "melspectrogram.onnx"),
    embedding_model_path=os.path.join(resources, "embedding_model.onnx"),
)
key = next(iter(oww.models))


def read_clip(path):
    sr, audio = scipy.io.wavfile.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if audio.dtype != np.int16:  # float WAVs are in [-1, 1]
        audio = np.clip(audio * (32767 if np.abs(audio).max() <= 1.0 else 1), -32768, 32767)
    if sr != SR:
        audio = scipy.signal.resample_poly(audio.astype(np.float32), SR, sr)
    return np.asarray(audio).astype(np.int16)


def windows(clip):
    """Classifier inputs (16 x 96) of the steps scoring >= THRESHOLD, over all offsets."""
    out = []
    for offset in OFFSETS:
        if offset >= len(clip):
            continue
        oww.reset()
        x = np.concatenate([np.zeros(WARMUP_STEPS * STEP, np.int16), clip[offset:]])
        for k in range(len(x) // STEP):
            score = float(oww.predict(x[k * STEP:(k + 1) * STEP])[key])
            if k + 1 >= WARMUP_STEPS and score >= THRESHOLD:
                out.append(oww.preprocessor.get_features(16)[0].astype(np.float32))
    return out


clips = sorted(glob.glob(os.path.join(output_dir, model_name, "positive_train", "*.wav")))
assert clips, f"no clips in {output_dir}/{model_name}/positive_train -- run Step 1 first"
rng = random.Random(0)
rng.shuffle(clips)

# Spread the cap across as many speakers as possible: at most a few windows per clip.
per_clip = max(1, MAX_WINDOWS // len(clips)) if len(clips) < MAX_WINDOWS else 1
picked, missed, used = [], 0, 0
for path in tqdm(clips, desc="Featurizing other speakers"):
    found = windows(read_clip(path))
    missed += not found
    used += bool(found)
    picked += rng.sample(found, min(per_clip, len(found)))
    if len(picked) >= MAX_WINDOWS:
        break
picked = picked[:MAX_WINDOWS]
assert picked, "no clip reached the threshold -- check the trained model"

out_path = os.path.join(output_dir, f"{model_name}_other_speakers.f32")
np.stack(picked).astype("<f4").tofile(out_path)
print(f"{len(picked)} windows from {used} clips ({missed} clips never reached {THRESHOLD}) -> "
      f"{out_path} ({os.path.getsize(out_path) / 1e6:.2f} MB)")

In [ ]:
# Step 6 (Optional): download the trained model and the voice match set for the TalkEasy app
#
# In the app repo: copy both into assets/wake_word/, list the .f32 in pubspec.yaml, and set
#   "classifier": "assets/wake_word/<model_name>.onnx",
#   "otherSpeakers": "assets/wake_word/<model_name>_other_speakers.f32"
# in assets/wake_word/wake_word.json.
import os

output_dir = config.get("output_dir", "./my_custom_model")
files_to_get = [os.path.join(output_dir, f"{config['model_name']}.onnx"),
                os.path.join(output_dir, f"{config['model_name']}_other_speakers.f32")]
try:
    from google.colab import files
    for path in files_to_get:
        files.download(path)
except ImportError:
    print("Not on Colab; the files are at:", *files_to_get, sep="\n  ")

After the model finishes training, the auto training script saves it as `my_custom_model/<model_name>.onnx` in the present working directory, where `<model_name>` is defined in the YAML training config file. It can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!